# Shadow Pancreas — Türkçe yemek adları için embedding karşılaştırması

**Amaç:** Menü dosyasındeki yemek adlarını (ör. *"Domates Çorba (Kaşarlı)"*, *"Kol Böreği"*, *"Şöbiyet"*) standart yemeklere bağlamak ve besin değeri bilinmeyen yemeklere en yakın yemeği bulmak için hangi embedding modelinin işe yaradığını ölçmek.

**Veri:** Tamamen **sentetik**: 127 standart yemek, 702 yazım varyantı/eş ad sorgusu, 38 elle işaretlenmiş zor çift. Besin değerleri porsiyon başına *yaklaşık* değerler, yalnızca bu deney için. Asıl sistemdeki değerler TürKomp + USDA FDC ile hesaplanacak.

| Görev | Soru | Metrik |
|---|---|---|
| **T1 Ad eşleştirme** | "Ezo Gelin Çorbası", "Koy Bazlamsi", "Spagetti (Napoliten)" → doğru standart yemek mi? | Acc@1, Recall@5, MRR |
| **T2 Karbonhidrat kNN** | Bir yemeği veritabanından çıkar; en yakın 3 komşunun karbonhidratıyla tahmin et | MAE (g/porsiyon), kategori isabeti |
| **T3 Zor çiftler** | *Ekmek ~ Bazlama* yakın, *Ekmek ~ Ekmek Kadayıfı* uzak çıkıyor mu? | ROC-AUC (0.5 = rastgele) |
| **T4 Besin korelasyonu** | Kosinüs benzerliği besin benzerliğiyle ne kadar örtüşüyor? | Spearman ρ |

T2–T4 iki biçimde ölçülür: **ad** (yalnızca yemek adı) ve **zengin** (ad + kısa malzeme açıklaması; gerçek sistemde bu açıklamayı yerel LLM üretecek).

**Çalıştırma:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`, ardından `Tümünü çalıştır`.

**EmbeddingGemma** gated bir model. Önce [huggingface.co/google/embeddinggemma-300m](https://huggingface.co/google/embeddinggemma-300m) sayfasında lisansı kabul et, sonra Colab'da soldaki anahtar simgesinden (*Secrets*) `HF_TOKEN` adıyla token'ını ekle ve "not defteri erişimi"ni aç. Token yoksa bu model atlanır, diğerleri çalışır.

Her embedding modeli **ayrı bir süreçte** çalışır. Bir model GPU'da çökerse diğerleri etkilenmez ve o model bir kez de CPU'da denenir.

In [ ]:
# transformers 5.x bazı modellerin kendi koduyla (gte-multilingual, embeddingmagibu) uyumsuz -> 4.x'e sabitle
!pip -q install "transformers>=4.56,<5" "sentence-transformers>=5.1,<6"
import sentence_transformers, transformers, torch
print("sentence-transformers", sentence_transformers.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "yok (CPU'da da çalışır, daha yavaş)")

In [ ]:
# İsteğe bağlı: gated modeller (EmbeddingGemma) için Hugging Face girişi
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
    print("HF girişi yapıldı")
except Exception as e:
    print("HF_TOKEN yok, gated modeller atlanabilir:", type(e).__name__)

In [ ]:
%%writefile synthetic_data.py
"""Sentetik yemek veri seti — YALNIZCA embedding karşılaştırması için.

Besin değerleri standart porsiyon başına YAKLAŞIK değerlerdir (genel beslenme
bilgisine dayalı elle girildi). Asıl sistemde kullanılmayacak; asıl değerler
TürKomp + USDA FoodData Central üzerinden tarif ayrıştırma ile hesaplanacak.

Alanlar: ad, kategori, karbonhidrat(g), protein(g), yağ(g), lif(g), GI sınıfı (D/O/Y),
kısa malzeme açıklaması, eş adlar (menülerde görülen yazımlar).
"""
import random
import re

# (ad, kategori, carb, prot, yag, lif, gi, aciklama, [es_adlar])
DISHES = [
    # --- ekmek / hamur işi ---
    ("Beyaz Ekmek", "ekmek_hamur", 25, 4, 1, 1.4, "Y", "buğday unu, maya, tuz; 2 dilim", ["Ekmek", "Francala", "Somun Ekmek"]),
    ("Tam Buğday Ekmeği", "ekmek_hamur", 21, 4.5, 1.5, 3.5, "O", "tam buğday unu, kepek, maya", ["Kepekli Ekmek", "Tam Buğday Ekmek"]),
    ("Bazlama", "ekmek_hamur", 30, 5, 1.5, 1.5, "Y", "buğday unu, yoğurt, maya; sacda pişmiş yassı ekmek", ["Köy Bazlaması", "Bazlama Ekmeği"]),
    ("Lavaş", "ekmek_hamur", 27, 4.5, 1, 1.2, "Y", "buğday unu, su, tuz; ince yassı ekmek", ["Lavaş Ekmeği", "Tandır Lavaş"]),
    ("Pide Ekmeği", "ekmek_hamur", 31, 5, 1.5, 1.4, "Y", "buğday unu, maya, çörek otu, susam", ["Ramazan Pidesi", "Pide"]),
    ("Simit", "ekmek_hamur", 55, 10, 6, 2.5, "Y", "buğday unu, pekmez, susam", ["Susamlı Simit", "Gevrek"]),
    ("Poğaça", "ekmek_hamur", 30, 5, 12, 1, "Y", "un, tereyağı, yoğurt, peynir", ["Peynirli Poğaça", "Sade Poğaça"]),
    ("Açma", "ekmek_hamur", 32, 5, 14, 1, "Y", "un, margarin, süt, maya", ["Sade Açma", "Zeytinli Açma"]),
    ("Kol Böreği", "ekmek_hamur", 28, 8, 15, 1, "O", "yufka, beyaz peynir, yağ", ["Peynirli Kol Böreği", "Kol Böreği (Peynirli)"]),
    ("Sigara Böreği", "ekmek_hamur", 18, 6, 14, 0.7, "O", "yufka, beyaz peynir, maydanoz; kızartma", ["Peynirli Sigara Böreği", "Sigara Börek"]),
    ("Su Böreği", "ekmek_hamur", 30, 12, 16, 1, "O", "haşlanmış yufka, peynir, tereyağı", ["Peynirli Su Böreği"]),
    ("Gül Böreği", "ekmek_hamur", 27, 8, 15, 1, "O", "yufka, peynir veya ıspanak", ["Peynirli Gül Böreği", "Ispanaklı Gül Böreği"]),
    ("Kaşarlı Tost", "ekmek_hamur", 30, 14, 13, 1.5, "Y", "tost ekmeği, kaşar peyniri", ["Tost", "Karışık Tost"]),
    ("Gözleme", "ekmek_hamur", 45, 12, 14, 2, "Y", "yufka, peynir veya patates; sacda", ["Peynirli Gözleme", "Patatesli Gözleme"]),
    # --- kahvaltılık ---
    ("Beyaz Peynir", "kahvaltilik", 0.5, 5, 6, 0, "D", "inek sütü peyniri; 30 g", ["Peynir", "Ezine Peyniri"]),
    ("Kaşar Peyniri", "kahvaltilik", 0.5, 8, 9, 0, "D", "olgunlaştırılmış kaşar; 30 g", ["Kaşar", "Taze Kaşar"]),
    ("Zeytin", "kahvaltilik", 1.5, 0.4, 6, 1.3, "D", "siyah veya yeşil zeytin; 10 adet", ["Siyah Zeytin", "Yeşil Zeytin"]),
    ("Haşlanmış Yumurta", "kahvaltilik", 0.6, 6.5, 5, 0, "D", "1 adet haşlanmış yumurta", ["Yumurta", "Haşlama Yumurta"]),
    ("Omlet", "kahvaltilik", 1.5, 13, 15, 0, "D", "2 yumurta, tereyağı", ["Yumurta, Omlet", "Sade Omlet"]),
    ("Menemen", "kahvaltilik", 7, 13, 17, 2, "D", "yumurta, domates, biber, zeytinyağı", ["Yumurta, Menemen", "Domatesli Yumurta"]),
    ("Bal", "kahvaltilik", 16, 0, 0, 0, "Y", "süzme çiçek balı; 1 yemek kaşığı", ["Süzme Bal", "Çiçek Balı"]),
    ("Reçel", "kahvaltilik", 13, 0, 0, 0.2, "Y", "meyve ve şeker; 1 yemek kaşığı", ["Vişne Reçeli", "Çilek Reçeli"]),
    ("Tahin Pekmez", "kahvaltilik", 15, 3, 8, 1, "O", "susam ezmesi ve üzüm pekmezi", ["Tahin-Pekmez", "Tahinli Pekmez"]),
    ("Tereyağı", "kahvaltilik", 0, 0, 8, 0, "D", "10 g tereyağı", ["Tereyağ"]),
    ("Sosis", "kahvaltilik", 2, 6, 12, 0, "D", "piliç veya dana sosis; 2 adet", ["Sosis Kızartma", "Piliç Sosis"]),
    ("Patates Kızartması", "kahvaltilik", 35, 3.5, 15, 3, "Y", "kızartılmış patates; 100 g", ["Patates Kızartma", "Parmak Patates", "Elma Dilim Patates"]),
    ("Domates Salatalık", "kahvaltilik", 4, 1, 0.2, 1.3, "D", "çeri domates ve salatalık dilimleri", ["Çeri Domates", "Salatalık Domates"]),
    ("Fındık Ezmesi", "kahvaltilik", 11, 1.5, 6, 1, "O", "kakaolu fındık kreması; 20 g", ["Kakaolu Fındık Kreması", "Çikolata Kreması"]),
    # --- çorba (1 kase) ---
    ("Mercimek Çorbası", "corba", 20, 8, 5, 4, "O", "kırmızı mercimek, soğan, havuç, un", ["Ev Usulü Mercimek Çorba", "Mercimek Çorba", "Süzme Mercimek Çorbası"]),
    ("Ezogelin Çorbası", "corba", 22, 7, 5, 4, "O", "kırmızı mercimek, bulgur, pirinç, nane", ["Ezogelin Çorba", "Ezo Gelin Çorbası"]),
    ("Tarhana Çorbası", "corba", 18, 5, 5, 1.5, "O", "tarhana (un, yoğurt, sebze), salça", ["Tarhana Çorba", "Ev Tarhanası"]),
    ("Yayla Çorbası", "corba", 14, 6, 6, 0.5, "O", "yoğurt, pirinç, yumurta, nane", ["Yayla Çorba", "Yoğurtlu Pirinç Çorbası"]),
    ("Domates Çorbası", "corba", 14, 3, 6, 1.5, "O", "domates, un, süt, rendelenmiş kaşar", ["Kaşarlı Domates Çorbası", "Domates Çorba (Kaşarlı)"]),
    ("Şehriye Çorbası", "corba", 16, 4, 4, 1, "Y", "tel şehriye, domates, tavuk suyu", ["Domatesli Şehriye Çorba", "Tel Şehriye Çorbası"]),
    ("Mantar Çorbası", "corba", 10, 3, 8, 1, "D", "mantar, krema, un", ["Kremalı Mantar Çorbası"]),
    ("Sebze Çorbası", "corba", 12, 3, 5, 3, "D", "havuç, kabak, patates, bezelye", ["Kremalı Taneli Sebze Çorbası", "Taze Sebze Çorbası"]),
    ("Tavuk Suyu Çorbası", "corba", 10, 8, 4, 0.5, "O", "tavuk suyu, tavuk eti, şehriye, limon", ["Tavuk Çorbası", "Terbiyeli Tavuk Suyu Çorba"]),
    ("Düğün Çorbası", "corba", 8, 8, 7, 0.3, "D", "kuzu eti suyu, yoğurt, yumurta terbiyesi", ["Düğün Çorba"]),
    ("Acem Çorbası", "corba", 12, 6, 6, 1.5, "O", "et suyu, pirinç, nohut, havuç", ["Acem Çorba"]),
    ("İşkembe Çorbası", "corba", 6, 10, 8, 0, "D", "işkembe, sarımsak, sirke", ["İşkembe Çorba"]),
    ("Brokoli Çorbası", "corba", 10, 4, 7, 3, "D", "brokoli, krema, patates", ["Kremalı Brokoli Çorbası"]),
    ("Naneli Yoğurt Çorbası", "corba", 13, 6, 6, 0.5, "O", "yoğurt, buğday, nane", ["Yoğurt Çorbası", "Naneli Yoğurt Çorba"]),
    # --- et ana yemek ---
    ("Izgara Köfte", "et_ana", 6, 22, 18, 0.5, "D", "kıyma, ekmek içi, soğan; ızgara", ["Köfte", "Kasap Köfte", "Izgara Köfte (Piyazlı)"]),
    ("Sahan Köfte", "et_ana", 10, 20, 18, 1.5, "D", "kıyma, domates sos, biber", ["Sahanda Köfte"]),
    ("İzmir Köfte", "et_ana", 20, 18, 18, 3, "O", "kıyma, patates, domates, biber; fırında", ["İzmir Köfte (Patatesli)", "Fırında Patatesli Köfte"]),
    ("Dana Rosto", "et_ana", 4, 25, 12, 0.5, "D", "dana eti, sebze sosu", ["Rosto Dana", "Dana Fleminyon"]),
    ("Tas Kebabı", "et_ana", 10, 22, 14, 2, "D", "kuşbaşı dana, soğan, salça", ["Tas Kebap", "Tas Kebap (Püreli)"]),
    ("Et Sote", "et_ana", 6, 22, 13, 1.5, "D", "kuşbaşı et, biber, domates", ["Dana Sote", "Etli Sote"]),
    ("Hünkar Beğendi", "et_ana", 20, 22, 22, 2, "O", "kuzu yahni, patlıcanlı beşamel", ["Hünkar Beğendili Kebap"]),
    ("Karnıyarık", "et_ana", 14, 12, 20, 5, "D", "patlıcan, kıyma, domates", ["Karnı Yarık"]),
    ("Musakka", "et_ana", 15, 12, 18, 5, "D", "patlıcan, kıyma, domates", ["Patlıcan Musakka", "Karışık Musakka"]),
    ("Hamburger", "et_ana", 35, 20, 18, 2, "Y", "hamburger ekmeği, köfte, sos", ["Burger", "Hamburger Menü"]),
    ("Lahmacun", "et_ana", 40, 14, 10, 3, "Y", "ince hamur, kıyma, soğan, biber", ["Fındık Lahmacun"]),
    ("Mantı", "et_ana", 50, 15, 15, 2, "O", "hamur, kıyma, yoğurt, tereyağı", ["Kayseri Mantısı", "Yoğurtlu Mantı"]),
    ("Pizza", "et_ana", 55, 18, 18, 3, "Y", "hamur, domates sos, mozzarella, sucuk", ["Karışık Pizza", "Margarita Pizza"]),
    # --- tavuk ana yemek ---
    ("Tavuk Sote", "tavuk_ana", 8, 25, 10, 1.5, "D", "tavuk göğsü, biber, domates, mantar", ["Piliç Sote", "Sebzeli Piliç Sote"]),
    ("Izgara Tavuk", "tavuk_ana", 1, 30, 7, 0, "D", "ızgara tavuk göğsü", ["Piliç Izgara", "Tavuk Göğüs Izgara"]),
    ("Tavuk Şiş", "tavuk_ana", 3, 28, 9, 0.5, "D", "şişte tavuk, biber, soğan", ["Piliç Şiş", "Tavuk Şiş Kebap"]),
    ("Fırın Tavuk Baget", "tavuk_ana", 2, 26, 14, 0, "D", "fırında tavuk but", ["Fırında Tavuk But", "Piliç Baget"]),
    ("Tavuk Döner", "tavuk_ana", 3, 25, 12, 0.5, "D", "döner tavuk eti (ekmeksiz)", ["Piliç Döner"]),
    ("Tavuk Şinitzel", "tavuk_ana", 18, 24, 15, 1, "O", "galeta unlu pane tavuk; kızartma", ["Tavuk Schnitzel", "Pane Tavuk"]),
    ("Çıtır Tavuk", "tavuk_ana", 20, 18, 16, 1, "Y", "panelenmiş tavuk parçaları; kızartma", ["Tavuk Nugget", "Çıtır Piliç"]),
    # --- sebze ana yemek ---
    ("Taze Fasulye", "sebze_ana", 10, 2, 8, 4, "D", "yeşil fasulye, domates, zeytinyağı", ["Zeytinyağlı Taze Fasulye", "Etli Taze Fasulye"]),
    ("Ispanak Yemeği", "sebze_ana", 7, 4, 6, 3.5, "D", "ıspanak, soğan, pirinç", ["Ispanak", "Yoğurtlu Ispanak"]),
    ("Türlü", "sebze_ana", 12, 3, 8, 4.5, "D", "patlıcan, kabak, biber, fasulye, domates", ["Sebze Türlü", "Etli Türlü"]),
    ("Kabak Yemeği", "sebze_ana", 8, 2, 6, 2, "D", "kabak, soğan, pirinç", ["Zeytinyağlı Kabak"]),
    ("Patates Yemeği", "sebze_ana", 25, 4, 7, 3, "Y", "patates, soğan, salça", ["Etli Patates", "Patates Oturtma"]),
    ("Patates Püresi", "sebze_ana", 22, 3, 6, 2, "Y", "haşlanmış patates, süt, tereyağı", ["Püre", "Tereyağlı Püre"]),
    ("Bezelye Yemeği", "sebze_ana", 15, 6, 6, 5, "D", "bezelye, havuç, patates", ["Etli Bezelye"]),
    ("Yaprak Sarma", "sebze_ana", 30, 3, 10, 3, "O", "asma yaprağı, pirinç, zeytinyağı", ["Zeytinyağlı Yaprak Sarma", "Sarma"]),
    ("Biber Dolması", "sebze_ana", 20, 8, 10, 3, "O", "dolmalık biber, pirinç, kıyma", ["Etli Biber Dolma", "Dolma"]),
    ("Mücver", "sebze_ana", 12, 6, 12, 2, "O", "kabak, yumurta, un, dereotu", ["Kabak Mücver"]),
    # --- baklagil ---
    ("Kuru Fasulye", "baklagil", 25, 10, 7, 9, "D", "kuru beyaz fasulye, salça, et", ["Etli Kuru Fasulye", "Kuru Fasulye Yemeği"]),
    ("Nohut Yemeği", "baklagil", 27, 10, 8, 8, "D", "nohut, salça, et", ["Nohut Yahni", "Etli Nohut"]),
    ("Barbunya", "baklagil", 22, 8, 7, 8, "D", "barbunya fasulyesi, havuç, zeytinyağı", ["Zeytinyağlı Barbunya", "Barbunya Pilaki"]),
    ("Yeşil Mercimek Yemeği", "baklagil", 25, 11, 6, 8, "D", "yeşil mercimek, bulgur, soğan", ["Mercimek Yemeği", "Bulgurlu Yeşil Mercimek"]),
    ("Mercimek Köftesi", "baklagil", 30, 8, 5, 6, "O", "kırmızı mercimek, ince bulgur, yeşillik", ["Mercimek Köfte"]),
    # --- pilav / makarna ---
    ("Pirinç Pilavı", "pilav_makarna", 45, 4, 7, 0.5, "Y", "pirinç, tereyağı, tavuk suyu", ["Pilav", "Tereyağlı Pirinç Pilavı", "Şehriyeli Pirinç Pilavı"]),
    ("Bulgur Pilavı", "pilav_makarna", 35, 5, 5, 6, "O", "pilavlık bulgur, salça, soğan", ["Salçalı Bulgur Pilavı", "Zerdeçallı Bulgur Pilavı", "Domatesli Bulgur Pilavı"]),
    ("Şehriye Pilavı", "pilav_makarna", 40, 6, 6, 1.5, "Y", "arpa şehriye, tereyağı", ["Arpa Şehriye Pilavı", "Tel Şehriye Pilavı"]),
    ("Kuskus Pilavı", "pilav_makarna", 38, 6, 5, 2, "O", "kuskus, tereyağı, sebze", ["Kuskus", "Sebzeli Kuskus"]),
    ("Meyhane Pilavı", "pilav_makarna", 33, 5, 6, 6, "O", "bulgur, domates, biber, soğan", ["Meyane Pilavı"]),
    ("Domates Soslu Makarna", "pilav_makarna", 45, 8, 6, 2.5, "O", "makarna, domates sos", ["Spagetti (Napoliten)", "Penne Arabiata", "Makarna"]),
    ("Kremalı Makarna", "pilav_makarna", 45, 10, 15, 2, "O", "makarna, krema, mantar, parmesan", ["Spagetti Carbonara", "Fettucini Alfredo", "Mantarlı Fettucini"]),
    ("Erişte", "pilav_makarna", 42, 8, 6, 2, "O", "ev yapımı yumurtalı erişte, peynir", ["Peynirli Erişte", "Ev Eriştesi"]),
    ("Fırın Makarna", "pilav_makarna", 40, 14, 14, 2, "O", "makarna, beşamel, kaşar", ["Beşamel Soslu Fırın Makarna"]),
    # --- şerbetli tatlı ---
    ("Baklava", "tatli_serbetli", 50, 5, 20, 1.5, "Y", "yufka, ceviz/fıstık, tereyağı, şerbet", ["Fıstıklı Baklava", "Cevizli Baklava", "Şöbiyet"]),
    ("Şekerpare", "tatli_serbetli", 55, 4, 15, 0.5, "Y", "irmik, un, tereyağı, şerbet", ["Kakaolu Şekerpare"]),
    ("Revani", "tatli_serbetli", 50, 4, 10, 0.5, "Y", "irmik, yumurta, şerbet", ["Portakallı Revani"]),
    ("Tel Kadayıf", "tatli_serbetli", 55, 5, 18, 1.5, "Y", "tel kadayıf, ceviz, tereyağı, şerbet", ["Burma Kadayıf", "Kadayıf"]),
    ("Ekmek Kadayıfı", "tatli_serbetli", 60, 4, 15, 0.5, "Y", "kadayıf ekmeği, şerbet, kaymak", ["Kaymaklı Ekmek Kadayıfı"]),
    ("Tulumba Tatlısı", "tatli_serbetli", 45, 2, 15, 0.3, "Y", "hamur kızartması, şerbet", ["Tulumba"]),
    ("Şam Tatlısı", "tatli_serbetli", 50, 4, 14, 0.5, "Y", "irmikli hamur, şerbet, fındık", ["Şam Tatlı"]),
    ("İrmik Helvası", "tatli_serbetli", 40, 4, 14, 1, "Y", "irmik, tereyağı, süt, şeker", ["Helva", "İrmik Helva"]),
    # --- sütlü tatlı / pasta ---
    ("Sütlaç", "tatli_sutlu", 35, 6, 5, 0.2, "O", "süt, pirinç, şeker; fırında", ["Fırın Sütlaç", "Sütlaç (Fırında)"]),
    ("Kazandibi", "tatli_sutlu", 35, 5, 5, 0, "O", "süt, nişasta, şeker; karamelize", ["Kazan Dibi"]),
    ("Tavuk Göğsü Tatlısı", "tatli_sutlu", 33, 6, 5, 0, "O", "süt, tavuk göğsü lifi, şeker, nişasta", ["Tavuk Göğsü", "Tavukgöğsü"]),
    ("Muhallebi", "tatli_sutlu", 30, 5, 5, 0, "O", "süt, pirinç unu, şeker", ["Sakızlı Muhallebi"]),
    ("Profiterol", "tatli_sutlu", 35, 5, 15, 0.5, "Y", "choux hamuru, krema, çikolata sos", ["Profiterol Tatlısı"]),
    ("Puding", "tatli_sutlu", 32, 5, 5, 1, "O", "süt, kakao, nişasta, şeker", ["Kakaolu Puding", "Supangle"]),
    ("Magnolya", "tatli_sutlu", 35, 5, 15, 0.5, "O", "bisküvi, krema, muhallebi, meyve", ["Çilekli Magnolya"]),
    ("Trileçe", "tatli_sutlu", 40, 7, 12, 0, "Y", "pandispanya, üç çeşit süt, karamel", ["Trileçe Tatlısı"]),
    ("Tiramisu", "tatli_sutlu", 30, 6, 18, 0.5, "O", "kedi dili, mascarpone, kahve", ["Tremisu"]),
    ("Aşure", "tatli_sutlu", 45, 5, 4, 4, "O", "buğday, nohut, fasulye, kuru meyve, şeker", ["Aşure Tatlısı"]),
    ("Kek", "tatli_sutlu", 35, 4, 12, 0.5, "Y", "un, şeker, yumurta, yağ", ["Kakaolu Kek", "Havuçlu Kek"]),
    ("Yaş Pasta", "tatli_sutlu", 40, 5, 18, 0.5, "Y", "pandispanya, krema, meyve", ["Pasta", "Kardelen Pasta"]),
    # --- meyve ---
    ("Elma", "meyve", 20, 0.3, 0.3, 3.5, "D", "1 orta boy elma", ["Kırmızı Elma", "Yeşil Elma"]),
    ("Portakal", "meyve", 15, 1, 0.2, 3, "D", "1 orta boy portakal", ["Washington Portakal"]),
    ("Muz", "meyve", 27, 1.3, 0.4, 3, "O", "1 orta boy muz", ["Sarı Muz"]),
    ("Mandalina", "meyve", 18, 1.2, 0.3, 3, "D", "2 adet mandalina", ["Satsuma Mandalina"]),
    ("Üzüm", "meyve", 27, 1, 0.3, 1.4, "O", "1 su bardağı üzüm", ["Çekirdeksiz Üzüm"]),
    ("Karpuz", "meyve", 19, 1.5, 0.4, 1, "Y", "1 dilim karpuz", ["Karpuz Dilimi"]),
    ("Mevsim Meyvesi", "meyve", 18, 0.8, 0.3, 2.5, "D", "mevsimine göre tek porsiyon meyve", ["Meyve", "Meyve Tabağı"]),
    # --- süt ürünü ---
    ("Yoğurt", "sut_urun", 9, 7, 6, 0, "D", "200 g tam yağlı yoğurt", ["Ev Yoğurdu", "Kase Yoğurt"]),
    ("Ayran", "sut_urun", 6, 4, 3, 0, "D", "yoğurt, su, tuz; 200 ml", ["Yayık Ayranı"]),
    ("Süt", "sut_urun", 10, 6.5, 6, 0, "D", "200 ml tam yağlı süt", ["Sade Süt", "Günlük Süt"]),
    ("Cacık", "sut_urun", 6, 4, 4, 0.5, "D", "yoğurt, salatalık, sarımsak, nane", ["Cacık (Salatalıklı)"]),
    ("Kakaolu Süt", "sut_urun", 24, 6, 5, 1, "O", "süt, kakao, şeker; 200 ml", ["Çikolatalı Süt"]),
    # --- salata ---
    ("Mevsim Salata", "salata", 6, 1.5, 5, 2.5, "D", "marul, domates, salatalık, zeytinyağı", ["Salata", "Mevsim Yeşillikleri", "Çoban Salata"]),
    ("Piyaz", "salata", 20, 7, 8, 6, "D", "kuru fasulye, soğan, maydanoz, yumurta", ["Fasulye Piyazı"]),
    ("Patates Salatası", "salata", 25, 3, 7, 3, "Y", "haşlanmış patates, soğan, maydanoz", ["Patates Salata"]),
    ("Kısır", "salata", 30, 5, 7, 5, "O", "ince bulgur, salça, yeşillik", ["Bulgur Kısırı"]),
    ("Rus Salatası", "salata", 15, 3, 20, 2, "O", "patates, havuç, bezelye, mayonez", ["Amerikan Salatası"]),
    # --- içecek / ek ---
    ("Çay", "icecek", 0, 0, 0, 0, "D", "şekersiz demlik çay", ["Demleme Çay", "Siyah Çay"]),
    ("Şeker", "icecek", 10, 0, 0, 0, "Y", "2 küp kesme şeker", ["Kesme Şeker", "Toz Şeker"]),
    ("Meyve Suyu", "icecek", 22, 0.5, 0, 0.3, "Y", "200 ml paket meyve suyu", ["Portakal Suyu", "Vişne Suyu"]),
    ("Kola", "icecek", 26, 0, 0, 0, "Y", "250 ml şekerli gazlı içecek", ["Gazlı İçecek"]),
]

# Elle işaretlenmiş zor çiftler.
# label=1: kelime olarak benzemiyor ama besin değeri yakın  (modelin YAKIN bulması istenir)
# label=0: kelime olarak benziyor ama besin değeri farklı (modelin UZAK bulması istenir)
HARD_PAIRS = [
    ("Beyaz Ekmek", "Bazlama", 1), ("Beyaz Ekmek", "Lavaş", 1), ("Bazlama", "Pide Ekmeği", 1),
    ("Pirinç Pilavı", "Şehriye Pilavı", 1), ("Bulgur Pilavı", "Kuskus Pilavı", 1),
    ("Mercimek Çorbası", "Ezogelin Çorbası", 1), ("Baklava", "Tel Kadayıf", 1),
    ("Sütlaç", "Kazandibi", 1), ("Muhallebi", "Kazandibi", 1), ("Sütlaç", "Tavuk Göğsü Tatlısı", 1),
    ("Kuru Fasulye", "Nohut Yemeği", 1), ("Barbunya", "Kuru Fasulye", 1), ("Elma", "Portakal", 1),
    ("Yoğurt", "Ayran", 1), ("Omlet", "Haşlanmış Yumurta", 1), ("Beyaz Peynir", "Kaşar Peyniri", 1),
    ("Bal", "Reçel", 1), ("Izgara Tavuk", "Tavuk Şiş", 1), ("Erişte", "Domates Soslu Makarna", 1),
    ("Revani", "Şekerpare", 1), ("Meyve Suyu", "Kola", 1), ("Simit", "Gözleme", 1),
    ("Beyaz Ekmek", "Ekmek Kadayıfı", 0), ("Tavuk Göğsü Tatlısı", "Izgara Tavuk", 0),
    ("Kuru Fasulye", "Taze Fasulye", 0), ("Sütlaç", "Süt", 0), ("Patates Salatası", "Mevsim Salata", 0),
    ("Şehriye Çorbası", "Şehriye Pilavı", 0), ("Mercimek Köftesi", "Izgara Köfte", 0),
    ("Domates Çorbası", "Domates Salatalık", 0), ("Kakaolu Süt", "Süt", 0),
    ("Tavuk Suyu Çorbası", "Izgara Tavuk", 0), ("Kaşarlı Tost", "Kaşar Peyniri", 0),
    ("Şeker", "Şekerpare", 0), ("Patates Kızartması", "Elma", 0), ("Naneli Yoğurt Çorbası", "Ayran", 0),
    ("Pide Ekmeği", "Lahmacun", 0), ("Mercimek Çorbası", "Mercimek Köftesi", 0),
]

TR_UP = str.maketrans("ıi", "Iİ")
TR_LO = str.maketrans("Iİ", "ıi")
ASCII = str.maketrans("çğıöşüÇĞİÖŞÜ", "cgiosuCGIOSU")
SUFFIX = [("Çorbası", "Çorba"), ("Pilavı", "Pilav"), ("Böreği", "Börek"), ("Tatlısı", "Tatlı"),
          ("Yemeği", "Yemek"), ("Kebabı", "Kebap"), ("Ekmeği", "Ekmek"), ("Köftesi", "Köfte"),
          ("Salatası", "Salata"), ("Helvası", "Helva")]
GARNISH = [" (Yoğurtlu)", " (Salata ile)", " (Ev Usulü)", " (Mevsim Sebzeli)", " (Porsiyon)"]


def tr_lower(s):
    return s.translate(TR_LO).lower()


def tr_upper(s):
    return s.translate(TR_UP).upper()


def toggle_suffix(s):
    for a, b in SUFFIX:
        if s.endswith(a):
            return s[: -len(a)] + b
        if s.endswith(b):
            return s[: -len(b)] + a
    return None


def typo(s, rng):
    letters = [i for i, c in enumerate(s) if c.isalpha() and 0 < i < len(s) - 1]
    if len(letters) < 3:
        return None
    i = rng.choice(letters)
    if rng.random() < 0.5:
        return s[:i] + s[i + 1:]
    if s[i + 1].isalpha():
        return s[:i] + s[i + 1] + s[i] + s[i + 2:]
    return s[:i] + s[i + 1:]


def build(seed=42):
    """Döndürür: dishes (list[dict]), queries (list[dict]), pairs (list[dict])."""
    rng = random.Random(seed)
    dishes = []
    for i, (name, cat, carb, prot, fat, fib, gi, desc, aliases) in enumerate(DISHES):
        dishes.append(dict(id=i, name=name, category=cat, carb=carb, protein=prot, fat=fat,
                           fiber=fib, gi=gi, desc=desc, aliases=aliases))
    names_norm = {tr_lower(d["name"]): d["id"] for d in dishes}

    queries = []

    def add(text, d, kind):
        key = tr_lower(text).strip()
        owner = names_norm.get(key)
        if owner is not None:  # birebir bir yemeğin adı ise (kendisi ya da başkası) sorgu olarak alma
            return
        queries.append(dict(query=text, target_id=d["id"], kind=kind))

    for d in dishes:
        for a in d["aliases"]:
            add(a, d, "es_ad")
        base = d["name"]
        add(base.translate(ASCII), d, "ascii")
        ts = toggle_suffix(base)
        if ts:
            add(ts, d, "ek_degisimi")
        t = typo(base, rng)
        if t:
            add(t, d, "yazim_hatasi")
        add(base + rng.choice(GARNISH), d, "garnitur")
        if d["aliases"]:
            a = rng.choice(d["aliases"])
            t2 = typo(a.translate(ASCII), rng)
            if t2:
                add(t2, d, "es_ad+ascii+hata")

    # aynı (sorgu, hedef) tekrarlarını temizle
    seen, uniq = set(), []
    for q in queries:
        k = (tr_lower(q["query"]), q["target_id"])
        if k not in seen:
            seen.add(k)
            uniq.append(q)

    by_name = {d["name"]: d["id"] for d in dishes}
    pairs = [dict(a=by_name[a], b=by_name[b], label=l) for a, b, l in HARD_PAIRS]
    return dishes, uniq, pairs


if __name__ == "__main__":
    d, q, p = build()
    import collections
    print(len(d), "yemek |", len(q), "sorgu |", len(p), "zor çift")
    print(collections.Counter(x["kind"] for x in q))
    print(collections.Counter(x["category"] for x in d))

In [ ]:
%%writefile embed_eval.py
"""Türkçe yemek adları için embedding modeli karşılaştırması.

Görevler
  T1  Ad eşleştirme   : yazım varyantı / eş ad -> standart yemek adı (Acc@1, Recall@5, MRR)
  T2  Karbonhidrat kNN: veritabanında olmayan bir yemeğin karbonhidratını en yakın 3 komşudan
                        tahmin et (leave-one-out MAE, g/porsiyon) + en yakın komşu kategori isabeti
  T3  Zor çiftler     : "ekmek~bazlama" yakın, "ekmek~ekmek kadayıfı" uzak çıkmalı (ROC-AUC)
  T4  Besin korelasyonu: kosinüs benzerliği ile besin vektörü benzerliği arasındaki Spearman
T2-T4 iki metin biçimiyle ölçülür: yalnızca ad, ad + kısa malzeme açıklaması (zenginleştirilmiş).
"""
import gc
import time

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score

from synthetic_data import tr_lower

GI_NUM = {"D": 0.0, "O": 1.0, "Y": 2.0}

# Adaylar: (görünen ad, HF kimliği veya [alternatif kimlikler], ayarlar)
#   prefix_q / prefix_d / prefix_s : sorgu / doküman / simetrik görevlerde metne eklenecek önek
#   prompt_q / prompt_d / prompt_s : modelin kendi prompt adları (varsa önek yerine kullanılır)
#   trust : trust_remote_code
QWEN_Q = ("Instruct: Given a Turkish dish name from a menu (may contain typos or missing "
          "Turkish characters), retrieve the canonical name of the same dish\nQuery: ")
QWEN_S = "Instruct: Given a Turkish dish name, retrieve dishes with similar ingredients and nutrition\nQuery: "
DENSE_MODELS = [
    ("MiniLM-L12 çok dilli", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", {}),
    ("e5-small", "intfloat/multilingual-e5-small", dict(prefix_q="query: ", prefix_d="passage: ", prefix_s="query: ")),
    ("e5-base", "intfloat/multilingual-e5-base", dict(prefix_q="query: ", prefix_d="passage: ", prefix_s="query: ")),
    ("e5-large", "intfloat/multilingual-e5-large", dict(prefix_q="query: ", prefix_d="passage: ", prefix_s="query: ")),
    ("bge-m3", "BAAI/bge-m3", {}),
    ("gte-multilingual-base", "Alibaba-NLP/gte-multilingual-base", dict(trust=True)),
    ("Qwen3-Embedding-0.6B", "Qwen/Qwen3-Embedding-0.6B", dict(prefix_q=QWEN_Q, prefix_s=QWEN_S)),
    ("EmbeddingGemma-300m", "google/embeddinggemma-300m",
     dict(prompt_q="query", prompt_d="document", prompt_s="STS")),
    ("TR-MTEB turkish-embedding", "trmteb/turkish-embedding-model", {}),
    ("embeddingmagibu-200m", ["magibu/embeddingmagibu-200m", "alibayram/embeddingmagibu-200m"], dict(trust=True)),
    ("BERTurk NLI-STS", "emrecan/bert-base-turkish-cased-mean-nli-stsb-tr", {}),
]


# ---------------------------------------------------------------- kodlayıcılar
class LexicalEncoder:
    """TF-IDF taban çizgisi. analyzer='char_wb' harf n-gram, 'word' kelime."""

    def __init__(self, analyzer):
        self.name = "TF-IDF " + ("harf 2-4" if analyzer == "char_wb" else "kelime")
        self.analyzer = analyzer
        self.vec = None

    def fit(self, corpus):
        kw = dict(analyzer=self.analyzer, preprocessor=tr_lower)
        if self.analyzer == "char_wb":
            kw["ngram_range"] = (2, 4)
        self.vec = TfidfVectorizer(**kw).fit(corpus)

    def encode(self, texts, mode="s"):
        m = self.vec.transform(texts).toarray()
        n = np.linalg.norm(m, axis=1, keepdims=True)
        return m / np.maximum(n, 1e-12)


class DenseEncoder:
    def __init__(self, label, model_id, cfg, device=None):
        from sentence_transformers import SentenceTransformer
        self.name = label
        self.cfg = cfg
        ids = model_id if isinstance(model_id, list) else [model_id]
        last = None
        for mid in ids:
            try:
                self.model = SentenceTransformer(mid, device=device, trust_remote_code=cfg.get("trust", False))
                self.model_id = mid
                break
            except Exception as e:  # noqa: BLE001
                last = e
        else:
            raise last
        self.prompts = getattr(self.model, "prompts", {}) or {}

    def fit(self, corpus):
        pass

    def encode(self, texts, mode="s"):
        kw = dict(normalize_embeddings=True, batch_size=64, convert_to_numpy=True, show_progress_bar=False)
        pname = self.cfg.get("prompt_" + mode)
        if pname and pname in self.prompts:
            kw["prompt_name"] = pname
        elif self.cfg.get("prefix_" + mode):
            kw["prompt"] = self.cfg["prefix_" + mode]
        return self.model.encode(list(texts), **kw)

    def close(self):
        del self.model
        gc.collect()
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:  # noqa: BLE001
            pass


# ---------------------------------------------------------------- yardımcılar
def nutrition_matrix(dishes):
    X = np.array([[d["carb"], d["protein"], d["fat"], d["fiber"], GI_NUM[d["gi"]]] for d in dishes], float)
    Z = (X - X.mean(0)) / X.std(0)
    Z[:, 0] *= 2.0  # diyabet için karbonhidrat en önemli bileşen
    return Z


def nutrition_similarity(dishes):
    Z = nutrition_matrix(dishes)
    d = np.sqrt(((Z[:, None, :] - Z[None, :, :]) ** 2).sum(-1))
    return -d


def texts_for(dishes, enriched):
    if enriched:
        return [f'{d["name"]}: {d["desc"]}' for d in dishes]
    return [d["name"] for d in dishes]


# ---------------------------------------------------------------- görevler
def task_match(enc, dishes, queries):
    D = enc.encode([d["name"] for d in dishes], mode="d")
    Q = enc.encode([q["query"] for q in queries], mode="q")
    S = Q @ D.T
    tgt = np.array([q["target_id"] for q in queries])
    order = np.argsort(-S, axis=1)
    rank = np.array([np.where(order[i] == tgt[i])[0][0] + 1 for i in range(len(tgt))])
    per_kind = pd.Series(rank == 1).groupby([q["kind"] for q in queries]).mean()
    wrong = [(queries[i]["query"], dishes[tgt[i]]["name"], dishes[order[i, 0]]["name"])
             for i in np.where(rank != 1)[0]]
    return dict(acc1=(rank == 1).mean(), r5=(rank <= 5).mean(), mrr=(1 / rank).mean()), per_kind, wrong, S


def task_carb_knn(enc, dishes, enriched, k=3):
    E = enc.encode(texts_for(dishes, enriched), mode="s")
    S = E @ E.T
    np.fill_diagonal(S, -np.inf)
    carb = np.array([d["carb"] for d in dishes])
    cat = np.array([d["category"] for d in dishes])
    nn = np.argsort(-S, axis=1)[:, :k]
    w = np.take_along_axis(S, nn, 1)
    w = np.exp((w - w.max(1, keepdims=True)) * 20)  # yakın komşuya daha çok ağırlık
    est = (carb[nn] * w).sum(1) / w.sum(1)
    return dict(carb_mae=np.abs(est - carb).mean(), cat_hit1=(cat[nn[:, 0]] == cat).mean())


def task_pairs(enc, dishes, pairs, enriched):
    E = enc.encode(texts_for(dishes, enriched), mode="s")
    sims = np.array([E[p["a"]] @ E[p["b"]] for p in pairs])
    labels = np.array([p["label"] for p in pairs])
    return roc_auc_score(labels, sims), sims


def task_corr(enc, dishes, enriched, nsim):
    E = enc.encode(texts_for(dishes, enriched), mode="s")
    S = E @ E.T
    iu = np.triu_indices(len(dishes), 1)
    return spearmanr(S[iu], nsim[iu]).correlation


def baselines(dishes):
    carb = np.array([d["carb"] for d in dishes], float)
    cat = np.array([d["category"] for d in dishes])
    n = len(carb)
    glob = np.abs((carb.sum() - carb) / (n - 1) - carb).mean()
    est = []
    for i in range(n):
        m = (cat == cat[i]) & (np.arange(n) != i)
        est.append(carb[m].mean() if m.any() else carb.mean())
    return dict(global_mean_mae=glob, category_mean_mae=np.abs(np.array(est) - carb).mean())


def evaluate(enc, dishes, queries, pairs, nsim):
    t0 = time.time()
    row = dict(model=enc.name)
    m, per_kind, wrong, S = task_match(enc, dishes, queries)
    row.update({"T1_acc@1": m["acc1"], "T1_recall@5": m["r5"], "T1_mrr": m["mrr"]})
    for enriched, tag in [(False, "ad"), (True, "zengin")]:
        c = task_carb_knn(enc, dishes, enriched)
        row[f"T2_carbMAE_{tag}"] = c["carb_mae"]
        row[f"T2_kategori@1_{tag}"] = c["cat_hit1"]
        auc, sims = task_pairs(enc, dishes, pairs, enriched)
        row[f"T3_AUC_{tag}"] = auc
        row[f"T4_spearman_{tag}"] = task_corr(enc, dishes, enriched, nsim)
        if tag == "ad":
            pair_sims = sims
    row["süre_sn"] = time.time() - t0
    return row, per_kind, wrong, pair_sims, S


def composite(df):
    """Her metrikte sıralama yapıp ortalama sıra -> 'genel sıra' (küçük daha iyi)."""
    cols_hi = ["T1_acc@1", "T3_AUC_ad", "T3_AUC_zengin", "T4_spearman_zengin", "T2_kategori@1_zengin"]
    cols_lo = ["T2_carbMAE_zengin"]
    r = pd.concat([df[c].rank(ascending=False) for c in cols_hi] + [df[c].rank(ascending=True) for c in cols_lo], axis=1)
    return r.mean(1)


# ---------------------------------------------------------------- hibrit (RRF)
def rank_metrics(S, queries):
    """Sorgu x yemek skor matrisinden T1 metrikleri ve sorgu türüne göre Acc@1."""
    tgt = np.array([q["target_id"] for q in queries])
    order = np.argsort(-S, axis=1)
    rank = np.array([np.where(order[i] == tgt[i])[0][0] + 1 for i in range(len(tgt))])
    per_kind = pd.Series(rank == 1).groupby([q["kind"] for q in queries]).mean()
    return dict(acc1=(rank == 1).mean(), r5=(rank <= 5).mean(), mrr=(1 / rank).mean()), per_kind


def rrf(score_mats, k=60, weights=None):
    """Reciprocal Rank Fusion: her yöntemin sıralamasını 1/(k+sıra) ile toplar."""
    weights = weights or [1.0] * len(score_mats)
    out = np.zeros_like(score_mats[0], dtype=float)
    for S, w in zip(score_mats, weights):
        ranks = np.argsort(np.argsort(-S, axis=1), axis=1) + 1
        out += w / (k + ranks)
    return out

In [ ]:
%%writefile run_one.py
"""Tek bir embedding modelini AYRI SÜREÇTE değerlendirir.

Neden ayrı süreç: bir model CUDA 'device-side assert' verirse GPU bağlamı bozulur ve aynı
süreçteki sonraki tüm modeller de düşer. Her model kendi sürecinde çalışınca hata yayılmaz.

Kullanım: python run_one.py <DENSE_MODELS indeksi> <cuda|cpu> <çıktı klasörü>
Çıktı: <klasör>/<indeks>.pkl  (row, per_kind, wrong, pair_sims, S)  veya  <indeks>.err
"""
import pickle
import sys
import time
import traceback

idx, device, outdir = int(sys.argv[1]), sys.argv[2], sys.argv[3]

from synthetic_data import build  # noqa: E402
from embed_eval import DENSE_MODELS, DenseEncoder, evaluate, nutrition_similarity  # noqa: E402

label, mid, cfg = DENSE_MODELS[idx]
dishes, queries, pairs = build(seed=42)
nsim = nutrition_similarity(dishes)
try:
    t0 = time.time()
    enc = DenseEncoder(label, mid, cfg, device=device)
    load = time.time() - t0
    row, per_kind, wrong, pair_sims, S = evaluate(enc, dishes, queries, pairs, nsim)
    row.update(yükleme_sn=load, cihaz=device, model_id=enc.model_id,
               boyut=int(enc.encode(["test"]).shape[1]),
               parametre_M=round(sum(p.numel() for p in enc.model.parameters()) / 1e6))
    with open(f"{outdir}/{idx}.pkl", "wb") as f:
        pickle.dump(dict(row=row, per_kind=per_kind, wrong=wrong, pair_sims=pair_sims, S=S), f)
    print("OK", label, device)
except Exception as e:  # noqa: BLE001
    with open(f"{outdir}/{idx}.err", "w", encoding="utf-8") as f:
        f.write(f"{type(e).__name__}: {e}\n\n{traceback.format_exc()}")
    print("HATA", label, device, type(e).__name__, str(e)[:200])
    sys.exit(1)

## 1. Sentetik veri

In [ ]:
import importlib, pandas as pd, numpy as np
import synthetic_data, embed_eval
importlib.reload(synthetic_data); importlib.reload(embed_eval)
from synthetic_data import build
from embed_eval import *

dishes, queries, pairs = build(seed=42)
nsim = nutrition_similarity(dishes)
ddf = pd.DataFrame(dishes)
qdf = pd.DataFrame(queries).assign(hedef=lambda x: x.target_id.map(ddf.name))
pdf = pd.DataFrame(pairs).assign(a_ad=lambda x: x.a.map(ddf.name), b_ad=lambda x: x.b.map(ddf.name))
print(len(ddf), "yemek |", len(qdf), "sorgu |", len(pdf), "zor çift")
display(qdf.kind.value_counts().rename("sorgu türü"))
display(qdf.sample(12, random_state=1)[["query", "kind", "hedef"]])
display(ddf[["name", "category", "carb", "protein", "fat", "fiber", "gi", "desc"]].head(10))
ddf.to_csv("sentetik_yemekler.csv", index=False); qdf.to_csv("sentetik_sorgular.csv", index=False); pdf.to_csv("zor_ciftler.csv", index=False)

## 2. Taban çizgileri (embedding'siz)

In [ ]:
B = baselines(dishes)
print(f"T2 karbonhidrat MAE | genel ortalama: {B['global_mean_mae']:.1f} g | kategori ortalaması (kategori biliniyorsa): {B['category_mean_mae']:.1f} g")

results, per_kind_all, wrong_all, pair_sims_all, S_all = [], {}, {}, {}, {}
lex_corpus = [d["name"] for d in dishes] + [a for d in dishes for a in d["aliases"]] + texts_for(dishes, True)
for an in ["char_wb", "word"]:
    enc = LexicalEncoder(an); enc.fit(lex_corpus)
    row, pk, wrong, ps, S = evaluate(enc, dishes, queries, pairs, nsim)
    results.append(row); per_kind_all[enc.name] = pk; wrong_all[enc.name] = wrong; pair_sims_all[enc.name] = ps; S_all[enc.name] = S
pd.DataFrame(results).round(3)

## 3. Embedding modelleri
Her model ayrı bir süreçte yüklenir ve ölçülür. GPU'da hata veren model bir kez CPU'da tekrar denenir. Yine de çalışmayan modelin hata nedeni yazdırılır.

In [ ]:
import subprocess, pickle, os, sys, time
OUT = "sonuclar_tmp"; os.makedirs(OUT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
skipped = {}
for i, (label, mid, cfg) in enumerate(DENSE_MODELS):
    for f in (f"{OUT}/{i}.pkl", f"{OUT}/{i}.err"):
        if os.path.exists(f): os.remove(f)
    for dev in ([device, "cpu"] if device == "cuda" else ["cpu"]):
        t0 = time.time()
        p = subprocess.run([sys.executable, "run_one.py", str(i), dev, OUT], capture_output=True, text=True)
        if os.path.exists(f"{OUT}/{i}.pkl"):
            break
        print(f"  {label}: {dev} başarısız ({time.time()-t0:.0f}s)" + (" -> CPU'da tekrar deneniyor" if dev == "cuda" else ""))
    if not os.path.exists(f"{OUT}/{i}.pkl"):
        err = open(f"{OUT}/{i}.err", encoding="utf-8").read() if os.path.exists(f"{OUT}/{i}.err") else p.stderr[-2000:]
        skipped[label] = err.splitlines()[0][:200] if err else "bilinmeyen hata"
        print("ATLANDI", label, "->", skipped[label]); continue
    r = pickle.load(open(f"{OUT}/{i}.pkl", "rb"))
    row = r["row"]; results.append(row)
    per_kind_all[label] = r["per_kind"]; wrong_all[label] = r["wrong"]; pair_sims_all[label] = r["pair_sims"]; S_all[label] = r["S"]
    print(f"OK {label:28s} [{row['cihaz']}] T1@1={row['T1_acc@1']:.3f}  T2 MAE={row['T2_carbMAE_zengin']:.1f}  T3 AUC(zengin)={row['T3_AUC_zengin']:.2f}")
print("Atlananlar:", skipped)

## 4. Sonuç tablosu
`genel_sıra`: T1 Acc@1, T2 MAE (zengin), T2 kategori isabeti (zengin), T3 AUC (ad ve zengin), T4 ρ (zengin) metriklerindeki sıraların ortalaması. Küçük olan daha iyi.

In [ ]:
df = pd.DataFrame(results)
df["genel_sıra"] = composite(df)
df = df.sort_values("genel_sıra").reset_index(drop=True)
show = ["model", "cihaz", "genel_sıra", "T1_acc@1", "T1_recall@5", "T2_carbMAE_ad", "T2_carbMAE_zengin", "T2_kategori@1_zengin",
        "T3_AUC_ad", "T3_AUC_zengin", "T4_spearman_ad", "T4_spearman_zengin", "parametre_M", "boyut", "süre_sn"]
show = [c for c in show if c in df]
(df[show].style.format(precision=3)
   .background_gradient(subset=["T1_acc@1", "T3_AUC_ad", "T3_AUC_zengin", "T4_spearman_zengin"], cmap="Greens")
   .background_gradient(subset=["T2_carbMAE_zengin"], cmap="Greens_r"))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 2, figsize=(14, 9))
order = df.model
def bar(a, col, title, ref=None, lo=False):
    a.barh(order, df[col], color="#3b7a57"); a.invert_yaxis(); a.set_title(title)
    for r, lab in (ref or []):
        a.axvline(r, ls="--", c="#b5523b"); a.text(r, len(order) - 0.5, " " + lab, color="#b5523b", va="bottom")
bar(ax[0, 0], "T1_acc@1", "T1 Ad eşleştirme — Acc@1 (yüksek iyi)")
bar(ax[0, 1], "T2_carbMAE_zengin", "T2 Karbonhidrat kNN MAE, g (düşük iyi)",
    ref=[(B["global_mean_mae"], "genel ort."), (B["category_mean_mae"], "kategori ort.")])
bar(ax[1, 0], "T3_AUC_ad", "T3 Zor çiftler AUC — yalnızca ad (0.5 = rastgele)", ref=[(0.5, "rastgele")])
bar(ax[1, 1], "T4_spearman_zengin", "T4 Besin korelasyonu ρ — zengin metin")
plt.tight_layout(); plt.show()

## 4b. Hibrit ad eşleştirme (RRF)
TF-IDF harf n-gram ile her embedding modelinin sıralamaları Reciprocal Rank Fusion ile birleştirilir. TF-IDF yazım hatalarında, embedding eş adlarda güçlü olduğu için birleşimin ikisini de geçmesi beklenir.

In [ ]:
hyb_rows, hyb_kind = [], {}
lex = "TF-IDF harf 2-4"
dense_names = [x for x in S_all if not x.startswith("TF-IDF")]
for m in dense_names:
    for w in [0.5, 1.0, 2.0]:
        S = rrf([S_all[lex], S_all[m]], weights=[1.0, w])
        met, pk_h = rank_metrics(S, queries)
        name = f"RRF: TF-IDF + {m}" + ("" if w == 1.0 else f" (embedding ağırlığı {w:g})")
        hyb_rows.append({"yöntem": name, "T1_acc@1": met["acc1"], "T1_recall@5": met["r5"], "T1_mrr": met["mrr"]})
        hyb_kind[name] = pk_h
for m in [lex] + dense_names:
    met, _ = rank_metrics(S_all[m], queries)
    hyb_rows.append({"yöntem": m + " (tek başına)", "T1_acc@1": met["acc1"], "T1_recall@5": met["r5"], "T1_mrr": met["mrr"]})
hyb = pd.DataFrame(hyb_rows).sort_values("T1_acc@1", ascending=False).reset_index(drop=True)
display(hyb.style.format(precision=3).background_gradient(subset=["T1_acc@1"], cmap="Greens"))
best_h = hyb[hyb["yöntem"].str.startswith("RRF")].iloc[0]["yöntem"]
print("En iyi hibrit:", best_h)
display(pd.DataFrame({best_h: hyb_kind[best_h], lex: per_kind_all[lex]}).T.style.format("{:.2f}"))
hyb.to_csv("hibrit_sonuclari.csv", index=False)

## 5. Hangi tür sorguda hata yapıyor? (T1)

In [ ]:
pk = pd.DataFrame(per_kind_all).T.loc[df.model]
display(pk.style.format("{:.2f}").background_gradient(cmap="RdYlGn", vmin=0.3, vmax=1))

## 6. Zor çiftlerde benzerlikler (yalnızca ad)
Etiket 1: besin olarak yakın (yüksek benzerlik beklenir). Etiket 0: kelime olarak benzer ama besin olarak farklı (düşük benzerlik beklenir).

In [ ]:
top = list(df.model[:4]) + [m for m in ["TF-IDF harf 2-4"] if m not in list(df.model[:4])]
tbl = pdf[["a_ad", "b_ad", "label"]].copy()
for m in top:
    tbl[m] = np.round(pair_sims_all[m], 2)
display(tbl.sort_values("label", ascending=False).style.background_gradient(subset=top, cmap="Blues"))

## 7. En iyi modelin hataları (T1)

In [ ]:
best = df.model[0]
print("En iyi:", best, "| hatalı sorgu sayısı:", len(wrong_all[best]))
display(pd.DataFrame(wrong_all[best], columns=["sorgu", "doğru", "modelin seçtiği"]).head(30))

## 8. Karar notları
- **T1 (ad eşleştirme)** normalizasyon adımı için belirleyici. Harf n-gram TF-IDF burada güçlü bir rakip; bir embedding onu geçemiyorsa normalizasyonda **hibrit** (TF-IDF + embedding, RRF ile birleştirme) kullanılmalı.
- **T3 / T4** "ekmek ~ bazlama" sorununun ölçüsü. Embedding'in tek başına besin benzerliğini yakalaması beklenmez. Asıl benzerlik **besin vektöründen** gelecek; embedding yalnızca (a) adı standart yemeğe bağlamak ve (b) besin değeri bilinmeyen yemek için yedek tahmin üretmek için kullanılacak.
- **T2**'de bir embedding *kategori ortalaması* çizgisinin altına iniyorsa yedek tahminci olarak işe yarıyor demektir.
- Menüdeki yemek kümesi sabit (~570 ad). Embedding'ler bir kez hesaplanıp dosyaya yazılabilir, yani çalışma anında embedding sunucusu gerekmeyebilir. Gerekirse kazanan modelin GGUF sürümü `llama-server --embeddings` ile ayrı bir portta çalıştırılabilir.
- Seçimde hız ve boyut da önemli: farklar küçükse küçük modeli seç.

In [ ]:
df.to_csv("embedding_sonuclari.csv", index=False)
pk.to_csv("embedding_sorgu_turu.csv")
try:
    from google.colab import files
    for f in ["embedding_sonuclari.csv", "embedding_sorgu_turu.csv", "hibrit_sonuclari.csv"]:
        files.download(f)
except Exception:
    print("Colab dışında: CSV dosyaları çalışma klasörüne yazıldı.")